# Adversarial ML for Security

## What This Is
Malware authors and attackers can craft inputs specifically designed to evade ML-based detectors. Adversarial ML for security covers both offensive (how to evade) and defensive (how to harden) angles.

## Feature Space vs Problem Space
A critical distinction: adversarial examples in feature space (perturb the numerical features) don't always translate to valid malware. A valid adversarial malware sample must:
1. Produce adversarial feature values
2. Still be executable / valid PE
3. Still deliver the intended malicious payload

This *problem-space constraint* severely limits adversarial attacks on malware detectors compared to image classifiers.

## Practical Evasion Techniques
- **Header manipulation**: Modify PE header fields that aren't functionally required
- **Dead code insertion**: Add legitimate-looking imports to push import table features toward benign
- **Section padding**: Add padded sections to reduce entropy features
- **Packing/obfuscation**: Change the surface while preserving behavior

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Rebuild malware classifier from NB 03
FEATURE_NAMES = [
    'file_size', 'n_sections', 'entropy_code', 'entropy_data',
    'n_imports', 'n_mal_imports', 'has_packer_sig', 'timestamp_valid',
    'n_strings', 'n_ip_strings', 'n_url_strings', 'export_count', 'tls_callback'
]

def generate_pe_features(is_malware: bool, n: int) -> np.ndarray:
    samples = []
    for _ in range(n):
        if is_malware:
            row = [np.random.randint(20000,500000), np.random.randint(3,8),
                   np.random.uniform(6.5,7.9), np.random.uniform(5.0,7.5),
                   np.random.randint(5,30), np.random.randint(2,8),
                   np.random.binomial(1,0.7), np.random.binomial(1,0.3),
                   np.random.randint(50,500), np.random.randint(1,10),
                   np.random.randint(0,5), np.random.randint(0,3),
                   np.random.binomial(1,0.4)]
        else:
            row = [np.random.randint(50000,5000000), np.random.randint(4,12),
                   np.random.uniform(5.0,6.5), np.random.uniform(2.0,5.0),
                   np.random.randint(30,200), np.random.randint(0,2),
                   np.random.binomial(1,0.05), np.random.binomial(1,0.9),
                   np.random.randint(200,5000), np.random.randint(0,2),
                   np.random.randint(0,3), np.random.randint(0,50),
                   np.random.binomial(1,0.02)]
        samples.append(row)
    return np.array(samples, dtype=float)

X_b = generate_pe_features(False, 600)
X_m = generate_pe_features(True, 400)
X = np.vstack([X_b, X_m])
y = np.array([0]*600 + [1]*400)
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]
mu, s = X[:800].mean(0), X[:800].std(0) + 1
X_n = (X - mu) / s
X_tr_n, y_tr = X_n[:800], y[:800]
X_te_n, y_te = X_n[800:], y[800:]

w = np.zeros(X_tr_n.shape[1]); b = 0.0
for _ in range(300):
    e = sigmoid(X_tr_n @ w + b) - y_tr
    w -= 0.05 * (X_tr_n.T @ e) / len(y_tr) + 0.01 * w
    b -= 0.05 * e.mean()

print(f'Malware classifier accuracy: {((sigmoid(X_te_n @ w + b) > 0.5).astype(int) == y_te).mean():.3f}')


In [ ]:
# Adversarial evasion attack: gradient-based feature perturbation
# Goal: move malware samples below the detection threshold

def adversarial_evasion(x_mal: np.ndarray, w: np.ndarray, b: float,
                         mu: np.ndarray, s: np.ndarray,
                         constraints: Dict[str, Tuple],
                         lr: float = 0.1, n_steps: int = 50) -> np.ndarray:
    """
    Gradient descent to minimize malware prediction score.
    constraints: {feature_name: (min, max)} for problem-space validity.
    """
    x = x_mal.copy()
    
    for _ in range(n_steps):
        x_n = (x - mu) / s
        pred = sigmoid(x_n @ w + b)
        if pred < 0.5:
            break  # evasion successful
        
        # Gradient of prediction w.r.t. x (chain rule through normalization)
        grad_w = pred * (1 - pred) * w / s  # gradient w.r.t. unnormalized x
        
        # Step in the direction that reduces malware score
        x = x - lr * grad_w
        
        # Apply problem-space constraints (can't reduce packer sig below 0, etc.)
        for i, feat_name in enumerate(FEATURE_NAMES):
            if feat_name in constraints:
                lo, hi = constraints[feat_name]
                x[i] = np.clip(x[i], lo, hi)
    
    return x

# Constraints: attacker can modify these features without breaking the malware
constraints = {
    'file_size': (0, 1e7),        # can pad file
    'n_imports': (0, 500),        # can add fake imports
    'n_strings': (0, 50000),      # can add string decoys
    'timestamp_valid': (0, 1),    # can fix timestamp
    'has_packer_sig': (0, 1),     # can remove packer signature
    'entropy_code': (3.0, 8.0),   # limited by actual code entropy
}

# Get malware samples from test set
mal_te_idx = np.where(y_te == 1)[0][:50]
X_mal_te = X[800:][mal_te_idx]

# Attack each malware sample
evaded = 0
for x_mal in X_mal_te:
    x_adv = adversarial_evasion(x_mal, w, b, mu, s, constraints)
    x_adv_n = (x_adv - mu) / s
    if sigmoid(x_adv_n @ w + b) < 0.5:
        evaded += 1

print(f'Adversarial Evasion Attack:')
print(f'  Malware samples tested: {len(X_mal_te)}')
print(f'  Successfully evaded: {evaded} ({evaded/len(X_mal_te):.1%})')
print(f'  Defense: adversarial training, feature ensembling, behavior-based fallback')
